# Get Features From Combine Tracking Data

We're checking if we can find novel insights about in-game player performance using Combine player-tracking data. We want to find new ways to examine players beyond drill outcomes such as forty yard dash time, vertical jump height, etc. 

In this file, we will try to extract features from the Combine player-tracking data that will help us predict if a defensive back will be successful at minimizing separation in man coverage situations.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================

import Data_Parameters as params

<br><br/>
#### Players Data (*BDB 2027*)

In [ ]:
# =============================
# Players Data
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Position of Interest
# -----------------------------

players_df = (
    players_df
    .filter(pl.col("nfl_position").is_in(["CB", "FS", "SS"]))
)

# -----------------------------
# Only Keep Needed Variables
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

# ----------------------------
# Identify 2023 Rookie DBs
# ----------------------------

rookie_db_2023 = (
    players_df
    .filter(pl.col("draft_year") == params.DRAFT_YEAR_OF_INTEREST)
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()
)

print("Players Data:")
print(f"    - Number of DBs From Combine: {len(players_of_interest)}")
print(f"    - Number of DBs From 2023 Rookie Class: {len(rookie_db_2023)}")
players_df

In [ ]:
# =============================
# Combine Tracking Data
# =============================

combine_tracking_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_tracking.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(
        pl.col("nfl_id").is_in(players_of_interest)
    )
)

# -----------------------------
# Don't Keep Ball Tracking Data
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("entity_type") == "PLAYER")
)

# ----------------------------
# Derive Velocity Variables (vx, vy)
# ----------------------------

combine_tracking_df = (
    combine_tracking_df
    .sort(["event_id", "time"])
    .with_columns(
        # Convert degrees to radians for trigonometry
        dir_rad = pl.col("dir") * (np.pi / 180)
    )
    .with_columns(
        # Reconstruct directional components of speed (velocity vectors)
        vx = pl.col("s") * pl.col("dir_rad").cos(),
        vy = pl.col("s") * pl.col("dir_rad").sin()
    )
)

# ---------------------------
# Calculate Linear Acceleration
# ---------------------------

combine_tracking_df = (
    combine_tracking_df
    # Ensure the data is strictly ordered by time within each drill event
    .sort(["event_id", "time"])
    .with_columns(
        linear_accel = (
            # (Current Speed - Previous Speed) / 0.1 seconds time delta
            (pl.col("s") - pl.col("s").shift(1).over("event_id")) / 0.1
        )
    )
)

print(f"Combine Tracking Data:")
combine_tracking_df

<br><br/>
## Choose Drills To Use
***

In [ ]:
# ==================================
# Find Percentage of Players of Interest
# With Tracking Data For Positional Drills
# ==================================

drill_tracking_summary_df = (
    combine_tracking_df
    .group_by("drill_name")
    .agg(
        num_w_atl_1_attempt = pl.col("nfl_id").n_unique()
    )
    .with_columns(
        perc_w_atl_1_attempt = (
            pl.col("num_w_atl_1_attempt")
            / combine_tracking_df.select(pl.col("nfl_id").n_unique()).item()
            * 100
        )
        .round(0)
    )
    .rename({"drill_name": "drill"})
    .sort("num_w_atl_1_attempt", descending = True)
)

print("Missingness in Position-Specific Drills:")
print(f"  - Drills With At Least 80% Participation (N = {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).height}): {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).select("drill").to_series().to_list()}")
drill_tracking_summary_df.head(10)

In [ ]:
# ===================================
# What Drills Have Sufficient Data?
# ===================================

# From the above we see the following: 

# Agility Drills That Have Good Exposure:
#     - W Drill Has 99% Exposure
#     - Box Drill Has 100% Exposure
#     - Teryl Austin Drill 1 has 81% Exposure
#     - 40-Yard Dash has 85% Exposure

# ====================================
# Drills That Could Be Useful
# ====================================

# FORTY_YARD_DASH
#   - ~88% exposure
#   - Clear choice for measuring burst, accel, top speed

# W_DRILL
#   - 99% exposure
#   - Good for repeated quick stops and starts

# BOX_DRILL
#   - 100% exposure
#   - Good measure for hip turn and lateral tracking

# TERYL_AUSTIN_DRILL_1
#   - ~94% exposure
#   - Good measure for click-and-close transition to follow WR

<br><br/>
## Derive Combine Performance Features
***

<br><br/>
#### Zero-To-Five Time

Zero-To-Five Time = The shortest amount of time it took the player to reach a speed of 5.5 y/s (~5 m/s) in their 40-yard dash attempts

In [ ]:
# ===========================================
# Get Explosive Ratio (Zero-To-Five Time)
#   - ACCELERATION FEATURE
# ===========================================

# -------------------------------------------
# For Each 40-Yard Dash, Calculate The Zero-To-Five Time
# -------------------------------------------

forty_dash_bursts = (
    combine_tracking_df
    # Only look at 40-yard dash drills
    .filter(pl.col("drill_name") == "FORTY_YARD_DASH")
    # For each drill...
    .group_by("event_id")
    # Identify the player, when they started moving, and when they reached 5.5y/s
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        # Time when they first started moving (speed > 0.2 y/s to ignore slight tracking jitter)
        start_time = pl.col("time").filter(pl.col("s") > 0.2).min(),
        # Time when they first crossed the 5.5 y/s threshold (equivalent to ~5 m/s)
        reached_five_time = pl.col("time").filter(pl.col("s") >= 5.5).min()
    )
    # Calculate zero-to-five time
    .with_columns(
        zero_to_five_time = (pl.col("reached_five_time") - pl.col("start_time")).dt.total_milliseconds() / 1000
    )
    # Filter out invalid zero-to-five times
    .filter(
        pl.col("zero_to_five_time").is_not_null() 
        & 
        (pl.col("zero_to_five_time") > 0)
    )
)

# -----------------------------------------
# For Each Player, Get Their Best Zero-To-Five Time
# -----------------------------------------

player_zero_to_five_df = (
    forty_dash_bursts
    .group_by("nfl_id")
    .agg(
        best_zero_to_five_time = pl.col("zero_to_five_time").min()
    )
    .sort("best_zero_to_five_time", "nfl_id")
)

print(f"Zero-To-Five Time Stats:")
player_zero_to_five_df

<br><br/>
#### Peak Jerk

Peak Jerk = The player's max change in linear acceleration during any 0.1 second period (given that it's at the start of their drill, before reaching 5.5 y/s) during their 40-yard dash attempts

In [ ]:
# ===========================================
# Get Peak Jerk
#   - ACCELERATION FEATURE
# ===========================================

# -------------------------------------------
# Calculate Jerk Frame-By-Frame During 40-Yard Dash Drills
# -------------------------------------------

jerk_df = (
    combine_tracking_df
    # Only look at 40-yard dash drills
    .filter(pl.col("drill_name") == "FORTY_YARD_DASH")
    .sort(["event_id", "time"])
    # Calculate jerk: (Current Accel - Previous Accel) / 0.1 seconds
    .with_columns(
        jerk = (
            (pl.col("linear_accel") - pl.col("linear_accel").shift(1).over("event_id")) / 0.1
        )
    )
)

# -------------------------------------------
# Isolate The Initial Start Pahse &
# Find The Max Jerk Per Attempt
# -------------------------------------------

attempt_jerk_df = (
    jerk_df
    # Focus only on the initial acceleration phase (while speed is building up to 5.5 y/s)
    .filter(pl.col("s") <= 5.5)
    .group_by("event_id")
    .agg(
        pl.col("nfl_id").first(),
        max_attempt_jerk = pl.col("jerk").max()
    )
)

# ------------------------------------------
# Get Best Peak Jerk For Each Player
# ------------------------------------------

player_peak_jerk_df = (
    attempt_jerk_df
    .group_by("nfl_id")
    .agg(
        best_peak_jerk = pl.col("max_attempt_jerk").max()
    )
    .sort("best_peak_jerk", "nfl_id")
)

print("Peak Jerk Stats:")
player_peak_jerk_df

<br><br/>
#### Click-and-Close Transition Time

Click-and-Close Transition Time = The minimum amount of time, from all their attempts in the TERYL_AUSTIN_DRILL_1 drill, for them to go from their slowest point during a CoD to reaching 4.5 y/s. This metric should quantify how quickly a DB can transition from a backpedal or deceleration phase into an aggressive forward drive. 

* Transition time = First timestep after a plant where speed >= 4.5 y/s - Timestep of the minimum speed recorded during the drill.

In [ ]:
# ==============================================================================
# Get Click-And-Close Transition Time (Teryl Austin Drill 1)
#   - REACTIVE TRANSITION FEATURE
# ==============================================================================

# ------------------------------------------------------------------------------
# For Each Drill Attempt, Calculate The Transition Delay from Plant to Drive
# ------------------------------------------------------------------------------

teryl_austin_transitions = (
    combine_tracking_df
    # 1. Filter for the specific drill
    .filter(pl.col("drill_name") == "TERYL_AUSTIN_DRILL_1")
    # 2. Sort by time within each attempt to ensure sequential analysis
    .sort("event_id", "time")
    # 3. For each attempt, isolate the plant moment and subsequent acceleration
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        
        # Find the absolute minimum speed during the change-of-direction window
        # (This represents the exact frame the feet plant to stop backpedaling)
        plant_time = pl.col("time").filter(
            (pl.col("s") == pl.col("s").min())
        ).first(),
        
        # Find the time when they successfully burst forward out of the plant
        # We look for the first time *after* the plant where speed climbs back over 4.5 y/s
        drive_time = pl.col("time").filter(
            (pl.col("time") > pl.col("time").filter(pl.col("s") == pl.col("s").min()).first()) & 
            (pl.col("s") >= 4.5)
        ).first()
    )
    # 4. Calculate the time elapsed to explosive drive phase
    .with_columns(
        click_and_close_time = (pl.col("drive_time") - pl.col("plant_time")).dt.total_milliseconds() / 1000
    )
    # 5. Clean up missing or invalid tracking rows
    .filter(
        pl.col("click_and_close_time").is_not_null() & 
        (pl.col("click_and_close_time") > 0)
    )
)

# ------------------------------------------------------------------------------
# For Each Player, Get Their Most Explosive (Minimum) Click-And-Close Time
# ------------------------------------------------------------------------------

player_click_close_df = (
    teryl_austin_transitions
    .group_by("nfl_id")
    .agg(
        best_click_and_close_time = pl.col("click_and_close_time").min()
    )
    .sort("best_click_and_close_time", "nfl_id")
)

print(f"Click-and-Close Transition Stats:")
player_click_close_df


<br><br/>
#### Deceleration-To-Acceleration Recovery

These three metrics - best_w_accel, best_w_decel, and best_w_darf - describe different aspects of a DBs CoD ability during the W drill.

best_w_accel = the max measured linear acceleration during their W drill attempts
* Higher value means DB has greater peak rate of speed increase

best_w_decel = the most negative linear acceleration during their W drill attempts
* Higher value means greater ability for DB to hit the breaks

best_w_darf = the greatest ratio of best_w_accel / best_w_decel during their W drill attempts
* Higher value means a higher acceleration relative to deceleration. Higher value isn't necessarily better.

In [ ]:
# ==============================================================================
# Get W-Drill Change of Direction Metrics
#   - BRAKING & RE-ACCELERATION FEATURE
# ==============================================================================

# ------------------------------------------------------------------------------
# For Each W-Drill Attempt, Calculate Peak Deceleration and Peak Acceleration
# ------------------------------------------------------------------------------

w_drill_metrics = (
    combine_tracking_df
    # 1. Only look at W-Drill attempts
    .filter(pl.col("drill_name") == "W_DRILL")
    # 2. Group by attempt to find the absolute limits of their movement
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        # Peak braking force (the most negative acceleration value)
        # We take the absolute value to represent it as a positive magnitude
        peak_w_decel = pl.col("linear_accel").min().abs(),
        # Peak burst force out of the cuts (the maximum positive acceleration value)
        peak_w_accel = pl.col("linear_accel").max()
    )
    # 3. Calculate the Deceleration-to-Acceleration Recovery Ratio (DARF)
    # This measures how efficiently they convert braking into forward explosive drive
    .with_columns(
        w_darf_ratio = pl.col("peak_w_accel") / pl.col("peak_w_decel")
    )
    # 4. Filter out any potential edge-case anomalies
    .filter(
        pl.col("peak_w_decel").is_not_null() & 
        (pl.col("peak_w_decel") > 0) &
        pl.col("peak_w_accel").is_not_null() & 
        (pl.col("peak_w_accel") > 0)
    )
)

# ------------------------------------------------------------------------------
# For Each Player, Get Their Best W-Drill Metrics Across All Attempts
# ------------------------------------------------------------------------------

player_w_drill_df = (
    w_drill_metrics
    .group_by("nfl_id")
    .agg(
        # We want the max capacity for both forces across their attempts
        best_w_decel = pl.col("peak_w_decel").max(),
        best_w_accel = pl.col("peak_w_accel").max(),
        best_w_darf  = pl.col("w_darf_ratio").max()
    )
    .sort("best_w_darf", descending=True)
)

print(f"W-Drill Performance Stats:")
player_w_drill_df

<br><br/>
#### Hip Fluidity Factor

best_box_peak_speed, best_box_turn_speed, and best_hip_fluidity_factor are complementary aspects of a DBs movement during the box drill (max speed, speed during slower portion of drill, and how that lower speed compares to their max speed)

best_box_peak_speed = The highest speed recorded during any of their box drill attempts

best_box_turn_speed = The highest 10th percentile speed recorded across their box drill attempts

best_hip_fluidity_factor = The highest attempt-level ratio of 10th percentile speed to peak speed

In [ ]:
# ==============================================================================
# Get Box Drill Hip Fluidity Metrics
#   - CENTRIPETAL VELOCITY MAINTENANCE FEATURE
# ==============================================================================

# ------------------------------------------------------------------------------
# For Each Box Drill Attempt, Calculate Peak Speed vs. Turn Speed
# ------------------------------------------------------------------------------

box_drill_metrics = (
    combine_tracking_df
    # 1. Only look at Box Drill attempts
    .filter(pl.col("drill_name") == "BOX_DRILL")
    # 2. Group by attempt to isolate their linear velocity vs. cornering velocity
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        # Peak absolute speed achieved during the straightaways of the figure-8
        peak_box_speed = pl.col("s").max(),
        # Cornering speed: The 10th percentile of speed across the drill.
        # This isolates their speed at the deepest, lowest velocity apex of the turns.
        apex_turn_speed = pl.col("s").quantile(0.10)
    )
    # 3. Calculate Velocity Maintenance Factor (Hip Fluidity Factor)
    # Higher percentage means they preserve speed efficiently through tight angles
    .with_columns(
        hip_fluidity_factor = pl.col("apex_turn_speed") / pl.col("peak_box_speed")
    )
    # 4. Filter out any potential invalid data
    .filter(
        pl.col("peak_box_speed").is_not_null() & 
        (pl.col("peak_box_speed") > 0)
    )
)

# ------------------------------------------------------------------------------
# For Each Player, Get Their Best Hip Fluidity Factor Across All Attempts
# ------------------------------------------------------------------------------

player_box_drill_df = (
    box_drill_metrics
    .group_by("nfl_id")
    .agg(
        best_box_peak_speed  = pl.col("peak_box_speed").max(),
        best_box_turn_speed  = pl.col("apex_turn_speed").max(),
        best_hip_fluidity_factor = pl.col("hip_fluidity_factor").max()
    )
    .sort("best_hip_fluidity_factor", descending=True)
)

print(f"Box Drill Hip Fluidity Stats:")
player_box_drill_df

<br><br/>
#### Combine All Metrics Into One Dataset

In [ ]:
# =======================================
# Dataset of All Combine Metrics
# =======================================

# Join All Tables Created Above So That We Have One 
# Table With All Combine Metrics For Each Player

combine_features_df = (
    players_df
    .select("nfl_id", "display_name")
    .join(player_zero_to_five_df, on = "nfl_id", how = "left")
    .join(player_peak_jerk_df, on = "nfl_id", how = "left")
    .join(player_click_close_df, on = "nfl_id", how = "left")
    .join(player_w_drill_df, on = "nfl_id", how = "left")
    .join(player_box_drill_df, on = "nfl_id", how = "left")
)

print("Combine Features Dataset:")
print(f"    - Dimensions: {combine_features_df.shape}")
combine_features_df

<br><br/>
#### Create *accel_score*

In [ ]:
# ===========================================
# Create Blended Master Agility Score Framework
# ===========================================

combine_features_df = (
    combine_features_df
    .with_columns(
        # -------------------------------------------
        # 1. Standardize Acceleration Metrics (0 to 1 Percentile)
        # -------------------------------------------
        # Lower time is better (descending rank)
        pct_start_time = (pl.col("best_zero_to_five_time").rank(descending=True) / pl.len()),
        # Higher jerk is better (ascending rank)
        pct_jerk = (pl.col("best_peak_jerk").rank(descending=False) / pl.len()),

        # -------------------------------------------
        # 2. Standardize Stopping Metrics (0 to 1 Percentile)
        # -------------------------------------------
        # Higher value means hits brakes harder (ascending rank)
        pct_w_decel = (pl.col("best_w_decel").rank(descending=False) / pl.len()),
    )
    .with_columns(
        # -------------------------------------------
        # 4. Calculate Blended Category Scores (Scaled to 0-100)
        # -------------------------------------------
        accel_score = (((pl.col("pct_start_time") + pl.col("pct_jerk")) / 2.0) * 100.0),
        decel_score = ((pl.col("pct_w_decel") / 1.0) * 100.0)
    )
)

print("Combine Metrics Dataset With Aggregated Scores:")
print(f"    - Number of Players: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

In [ ]:
# ===========================================
# Create Scores With Normalized Values
# Rather Than Percentile Ranks
# ===========================================

# combine_features_df = (
#     combine_features_df
#     .with_columns(
#         # -------------------------------------------
#         # 1. Normalize Acceleration Metrics (0 to 1)
#         # -------------------------------------------
#         # Lower time is better: reverse normalized value
#         norm_start_time = (
#             (pl.col("best_zero_to_five_time").max() - pl.col("best_zero_to_five_time"))
#             / (
#                 pl.col("best_zero_to_five_time").max()
#                 - pl.col("best_zero_to_five_time").min()
#             )
#         ),

#         # Higher jerk is better
#         norm_jerk = (
#             (pl.col("best_peak_jerk") - pl.col("best_peak_jerk").min())
#             / (
#                 pl.col("best_peak_jerk").max()
#                 - pl.col("best_peak_jerk").min()
#             )
#         ),

#         # -------------------------------------------
#         # 2. Normalize Deceleration Metric (0 to 1)
#         # -------------------------------------------
#         # Higher best_w_decel is better
#         norm_w_decel = (
#             (pl.col("best_w_decel") - pl.col("best_w_decel").min())
#             / (
#                 pl.col("best_w_decel").max()
#                 - pl.col("best_w_decel").min()
#             )
#         ),
#     )
#     .with_columns(
#         # -------------------------------------------
#         # 3. Calculate Category Scores (Scaled 0-100)
#         # -------------------------------------------
#         norm_accel_score = (
#             (pl.col("norm_start_time") + pl.col("norm_jerk")) / 2.0 * 100.0
#         ),

#         norm_decel_score = (
#             pl.col("norm_w_decel") * 100.0
#         ),
#     )
# )

# print("Combine Metrics Dataset With Aggregated Scores:")
# print(f"    - Number of Players: {combine_features_df.select('nfl_id').n_unique()}")

# combine_features_df


<br><br/>
#### Examine Missingness

In [ ]:
# ========================================
# Missingness In Combine Features Dataset
# ========================================

combine_metrics = [
    "best_zero_to_five_time", 
    "best_peak_jerk", 
    "best_click_and_close_time", 
    "best_w_decel", 
    "best_w_accel",	
    "best_w_darf", 
    "best_box_peak_speed", 
    "best_box_turn_speed", 
    "best_hip_fluidity_factor", 
]

print("Percentage of Players With Missing Values For Each Combine Metric:")
(
    combine_features_df
    .select(combine_metrics)
    .select(((pl.all().is_null().sum()) / players_df.height * 100).round(2))
    .transpose(include_header=True, header_name="feature")
    .rename({"column_0": "percent_missing"})
    .sort("percent_missing")
)

<br><br/>
## Export Data
***

In [ ]:
# ===================================
# Combine Features Data
# ===================================

# For all WRs in our BDB2027 Rookie Cohort, we derive some metrics
# related to their acceleration, deceleration, and cutting abilities. 
# These metrics are stored here. There's one row per player.

combine_features_df.write_csv(str(SEPARATION_DATA_DIR / "13_DB_Combine_Features_Data.csv"))